# 04 — EXPLAIN sweep (default schema)

Plans every `(SQL, params)` pair the three matrix reports recorded, against the
schema **as this cluster runs it**, and reports what scans what.

Companion to `03_api_index_matrix.ipynb`, which produced the reports. This
notebook drives no APIs and needs no Polaris restart.

**Why plain EXPLAIN.** It does not execute, which is what makes the write half
askable at all and leaves no clock to misquote. Plan *shape* is deterministic
here; execution time is not — the same plan measured 7.523 / 3.289 / 4.571 /
1.622 ms across repeats, a 4.6x spread that once moved a reported speedup from
11.6x to 6.3x on noise alone.

## This notebook does not change the schema

It creates no index and drops none. The question it answers is *what does this
deployment do*, and an experiment that adds an index is a different question —
one whose answer is worthless if the measurement it is compared against was
taken on a mutated schema.

Testing a proposed index is legitimate and deliberately lives elsewhere:
`drop_grantee_index.py` removes one, and a `CREATE INDEX CONCURRENTLY` puts it
back. Run those knowingly, from a shell, and re-run this notebook afterwards —
never as a side effect of the measurement.

## Run order

`Restart & Run All`. Read-only against the schema, so it is safe to re-run at
any time.


In [10]:
# --- cell 1: bootstrap ------------------------------------------------------
import gc
import json
import os
import pathlib
import shlex
import subprocess
import sys
from datetime import datetime

_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / "src"))

from nb_support import *  # noqa: F403 — init_env, root_token, PG_* globals

init_env("local")
require_not_prod("04: drops and recreates an index on grant_records")

#: Second guard, independent of require_not_prod. This notebook DROPS an index
#: on the table the whole audit measures. A wrong endpoint here is not a bad
#: measurement, it is a production change.
assert any(h in POLARIS_URL for h in ("localhost", "127.0.0.1", "192.168.")), (
    f"Refusing to run against {POLARIS_URL} — 04 is local-only."
)

HERE = pathlib.Path.cwd()
RUNS = HERE / "runs"
REPORTS = HERE / "reports"
RUNS.mkdir(exist_ok=True)

#: Cluster coordinates. `init_env` supplies PG_USER / PG_PASSWORD / PG_DB but
#: NOT these — they are kubectl's, not the database's. Same defaults and same
#: env-var overrides as 03, so the two notebooks address one cluster.
#: THE PRIMARY IS `-1`, NOT `-0`. Earlier manual resets hit a read-only replica
#: and reported success; a DROP INDEX against a replica fails outright, which is
#: the better failure but still the wrong pod.
NS = os.environ.get("POLARIS_K8S_NAMESPACE", "datahub-hynix")
PGPOD = os.environ.get("PGPOD", "benchmarks-postgresql-postgresql-ha-postgresql-1")

SCHEMA = "polaris_schema"
INDEX = "idx_grant_records_grantee"
CASES = ("unauthorized", "authorized", "admin")
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Postgres : {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"kubectl  : -n {NS} {PGPOD}")
print(f"stamp    : {STAMP}")


🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
Postgres : 192.168.139.2:5432/polaris
kubectl  : -n datahub-hynix benchmarks-postgresql-postgresql-ha-postgresql-1
stamp    : 20260903-142027


## Flags

`ARM_TOGGLE` is the only destructive switch. Leave it `False` to sweep the
current state alone.


In [11]:
# --- cell 2: flags ----------------------------------------------------------
#: Print the worklist before planning anything. The dry run touches no
#: database and is where refusals are read -- a refused pair is not planned and
#: not counted, and some refusals are parser faults worth stopping for.
DRY_RUN_FIRST = True

print(f"DRY_RUN_FIRST = {DRY_RUN_FIRST}")
print("schema is NOT modified by this notebook")


DRY_RUN_FIRST = True
schema is NOT modified by this notebook


In [12]:
# --- cell 3: helpers --------------------------------------------------------
def sh(cmd, check=True, quiet=False):
    """Run a shell command from HERE, stream it, raise on failure."""
    p = subprocess.run(cmd, shell=True, cwd=str(HERE), capture_output=True, text=True)
    if not quiet:
        if p.stdout:
            print(p.stdout.rstrip())
        if p.stderr:
            print(p.stderr.rstrip(), file=sys.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p


def psql(sql, quiet=True):
    """One statement through psql -tAc, on the PRIMARY.

    THE SQL IS FLATTENED TO ONE LINE FIRST. `kubectl exec` passes `\n` through
    literally and psql reads a leading backslash-n as a META-COMMAND, failing
    with `invalid command \nSELECT` — so a multi-line query silently becomes a
    psql syntax error rather than a result. Whitespace is nothing to SQL.
    """
    one_line = " ".join(sql.split())
    return sh(
        f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
        f"psql -U {PG_USER} -d {PG_DB} -tAc {shlex.quote(one_line)}",
        quiet=quiet,
    ).stdout.strip()


def index_present():
    return psql(
        f"SELECT count(*) FROM pg_indexes WHERE indexname = '{INDEX}'"
    ) == "1"


def latest_report(case):
    hits = sorted(REPORTS.glob(f"doc-api-sql-matrix-{case}-*.md"))
    return hits[-2] if hits else None


print("helpers defined")


helpers defined


## Preflight

Nothing is planned until every report is found and the live index state is
known. `explain_api_matrix.py` additionally **refuses a report that did not
fully parse**, before touching the database — a short worklist looks exactly
like a complete one.


In [13]:
# --- cell 4: preflight ------------------------------------------------------
START_STATE = "present" if index_present() else "absent"
OTHER_STATE = "absent" if START_STATE == "present" else "present"
print(f"live index state: {START_STATE}  ({INDEX})")

vol = psql(f"SELECT count(*) FROM {SCHEMA}.grant_records")
print(f"grant_records rows: {int(vol):,}")

MATRICES = {}
for case in CASES:
    r = latest_report(case)
    MATRICES[case] = r
    print(f"  {case:<13} {r.name if r else '!! NO REPORT — drive it first'}")

missing = [c for c, r in MATRICES.items() if r is None]
assert not missing, (
    f"no matrix report for {missing}. Drive those cases in 03 first — sweeping "
    "two of three states produces a contrast nobody can read."
)


live index state: absent  (idx_grant_records_grantee)
grant_records rows: 60,815
  unauthorized  doc-api-sql-matrix-unauthorized-explained-20260903-141115.md
  authorized    doc-api-sql-matrix-authorized-explained-20260903-141115.md
  admin         doc-api-sql-matrix-admin-explained-20260903-141115.md


In [14]:
# --- cell 5: dry run — the worklist, touching no database -------------------
if DRY_RUN_FIRST:
    for case in CASES:
        print(f"\n=== {case} ===")
        sh(f"uv run python explain_api_matrix.py --case {case} "
           f"--index-state {START_STATE} "
           f"--matrix {MATRICES[case].relative_to(HERE)} --dry-run")
else:
    print("DRY_RUN_FIRST is off")



=== unauthorized ===
reports/doc-api-sql-matrix-unauthorized-explained-20260903-141115.md
  43 APIs, 390 statement instances
  33 (SQL, params) pairs over 11 distinct SQL texts
  33 replayable, 0 refused

=== authorized ===
reports/doc-api-sql-matrix-authorized-explained-20260903-141115.md
  43 APIs, 460 statement instances
  93 (SQL, params) pairs over 25 distinct SQL texts
  91 replayable, 2 refused
    - grant_records SELECT: no parameters were recorded for this statement
    - events INSERT: no parameters were recorded for this statement

=== admin ===
reports/doc-api-sql-matrix-admin-explained-20260903-141115.md
  43 APIs, 564 statement instances
  138 (SQL, params) pairs over 30 distinct SQL texts
  128 replayable, 10 refused
    - grant_records SELECT: placeholder/parameter mismatch: 13 placeholders, 5 values
    - grant_records SELECT: placeholder/parameter mismatch: 13 placeholders, 5 values
    - grant_records SELECT: placeholder/parameter mismatch: 13 placeholders, 5 values

**Read the refusals before continuing.** A refused pair is not planned and not
counted, which is the honest behaviour — but three of them are permanent and the
rest are not:

- `parameters were redacted at capture (secret table)` —
  `principal_authentication_data`, 3 pairs in the admin case. Permanent, by
  design.
- anything else — a parser fault. Stop and fix it rather than sweeping around
  it; the 2026-09-03 dry run found six corrupted `grant_records` statements
  this way.


In [15]:
# --- cell 6: sweep the state the cluster is already in ----------------------
for case in CASES:
    print(f"\n=== {case} · index {START_STATE} ===")
    sh(f"uv run python explain_api_matrix.py --case {case} "
       f"--index-state {START_STATE} "
       f"--matrix {MATRICES[case].relative_to(HERE)}")



=== unauthorized · index absent ===
reports/doc-api-sql-matrix-unauthorized-explained-20260903-141115.md
  43 APIs, 390 statement instances
  33 (SQL, params) pairs over 11 distinct SQL texts
  33 replayable, 0 refused

volume: entities=9,646, grant_records=60,815, policy_mapping_record=0, principal_authentication_data=1,108
index absent: 43 of 43 APIs sequentially scan something
  relations seq-scanned: grant_records
  -> apiexplain-unauthorized-20260903-142053.json

=== authorized · index absent ===
reports/doc-api-sql-matrix-authorized-explained-20260903-141115.md
  43 APIs, 460 statement instances
  93 (SQL, params) pairs over 25 distinct SQL texts
  91 replayable, 2 refused
    - grant_records SELECT: no parameters were recorded for this statement
    - events INSERT: no parameters were recorded for this statement

volume: entities=9,646, grant_records=60,815, policy_mapping_record=0, principal_authentication_data=1,108
index absent: 43 of 43 APIs sequentially scan something
  re

## Results

One run per case, all in the state the cluster is in. Each records the live
`pg_indexes` state it was taken in, so a run can never be mislabelled — and if
someone changes the schema between cases, the runs will disagree and say so.


In [16]:
# --- cell 7: what was produced ---------------------------------------------
import pandas as pd

#: NEWEST RUN PER (case, index state), superseded ones counted not listed.
#: Every sweep leaves another apiexplain-*.json behind, so a plain glob grows a
#: row per re-run: the first re-run of this notebook produced SIX rows for
#: three cases, all identical, and a reader cannot tell which one the finding
#: below came from. Old runs are kept on disk deliberately -- they are evidence
#: -- but only the current one belongs in the summary.
all_runs = []
for f in sorted(RUNS.glob("apiexplain-*.json")):
    d = json.loads(f.read_text())
    per = d.get("per_api", {})
    rels = sorted({x for r in per.values() for x in r.get("seq_scanned", [])})
    all_runs.append({
        "case": d.get("case"),
        "index": d.get("index_state"),
        "APIs": len(per),
        "pairs": d.get("pairs"),
        "seq-scanning APIs": sum(1 for r in per.values() if r.get("seq_scanned")),
        "relations seq-scanned": ", ".join(rels) or "—",
        "grant_records rows": (d.get("volume") or {}).get("grant_records"),
        "run": f.name,
    })

if not all_runs:
    print("no apiexplain-*.json yet")
    df = None
else:
    latest = {}
    for r in all_runs:                       # sorted by name == sorted by stamp
        latest[(r["case"], r["index"])] = r
    df = pd.DataFrame(latest.values()).sort_values(["case", "index"])
    display(df)
    superseded = len(all_runs) - len(latest)
    if superseded:
        print(f"({superseded} older run(s) on disk, not shown — kept as evidence)")
    have = set(latest)
    missing = [c for c in CASES if not any(h[0] == c for h in have)]
    if missing:
        print(f"\nno run for: {missing}")
    #: All three must have been planned in the SAME state, or the table
    #: compares schemas rather than identities.
    states = {h[1] for h in have}
    if len(states) > 1:
        print(f"\nRUNS SPAN MORE THAN ONE INDEX STATE: {sorted(states)} — read "
              "the `index` column before comparing any two rows.")


,case,index,APIs,pairs,seq-scanning APIs,relations seq-scanned,grant_records rows,run
0,admin,absent,43,138,43,grant_records,60815,apiexplain-admin-20260903-142056.json
1,authorized,absent,43,93,43,grant_records,60815,apiexplain-authorized-20260903-142055.json
2,unauthorized,absent,43,33,43,grant_records,60815,apiexplain-unauthorized-20260903-142053.json


(12 older run(s) on disk, not shown — kept as evidence)


In [17]:
# --- cell 8: the finding ----------------------------------------------------
#: What does this deployment do, as configured. There is no contrast to draw --
#: the notebook does not change the schema -- so the result is stated directly:
#: which (case, API) pairs scan something sequentially, on what, at the volume
#: stamped on the run.
runs = {}
for f in sorted(RUNS.glob("apiexplain-*.json")):
    d = json.loads(f.read_text())
    d["_file"] = f.name
    #: Keyed by (case, state), so a re-run REPLACES its predecessor rather than
    #: sitting beside it. Sorted glob order means the newest wins.
    runs[(d.get("case"), d.get("index_state"))] = d

rows = []
used = []
for (case, state), d in sorted(runs.items()):
    used.append(d["_file"])
    for api, r in sorted(d["per_api"].items()):
        rows.append({
            "case": case,
            "api": api,
            "seq-scans": ", ".join(r.get("seq_scanned") or []) or "—",
            "indexes used": ", ".join(r.get("indexes_used") or []) or "—",
            "planned": r.get("planned"),
            "skipped": r.get("skipped"),
        })

if not rows:
    print("no runs yet")
else:
    sdf = pd.DataFrame(rows)
    any_run = next(iter(runs.values()))
    vol = (any_run.get("volume") or {}).get("grant_records")
    state = any_run.get("index_state")
    seq = sdf[sdf["seq-scans"] != "—"]
    hot = sdf[sdf["seq-scans"].str.contains("grant_records")]
    print(f"schema as deployed (idx_grant_records_grantee: {state})"
          + (f" · grant_records = {vol:,} rows" if isinstance(vol, int) else ""))
    #: Name the evidence. Two runs 25 minutes apart produced identical numbers,
    #: which is worth knowing -- but only if the reader can tell WHICH run the
    #: table below was built from.
    print("from: " + ", ".join(used))
    print(f"{len(seq)} of {len(sdf)} (case, API) pairs sequentially scan something")
    print(f"{len(hot)} of {len(sdf)} scan grant_records — the authorization "
          f"prelude every request pays\n")
    display(seq if len(seq) else sdf.head(20))
    if not len(seq):
        print("Nothing scans sequentially. Check the volume above before "
              "reading that as a clean bill of health.")


schema as deployed (idx_grant_records_grantee: absent) · grant_records = 60,815 rows
from: apiexplain-admin-20260903-142056.json, apiexplain-authorized-20260903-142055.json, apiexplain-unauthorized-20260903-142053.json
129 of 129 (case, API) pairs sequentially scan something
129 of 129 scan grant_records — the authorization prelude every request pays



,case,api,seq-scans,indexes used,planned,skipped
0,admin,iceberg.commit_table,grant_records,"constraint_name, entities_pkey, idx_entities",13,0
1,admin,iceberg.create_namespace,grant_records,"constraint_name, entities_pkey, grant_records_...",16,0
2,admin,iceberg.create_table,grant_records,"constraint_name, entities_pkey, grant_records_...",16,0
3,admin,iceberg.create_view,grant_records,"constraint_name, entities_pkey, grant_records_...",16,0
4,admin,iceberg.drop_namespace,grant_records,"constraint_name, entities_pkey, grant_records_...",15,1
...,...,...,...,...,...,...
124,unauthorized,mgmt.list_grants,grant_records,"constraint_name, entities_pkey, idx_entities",8,0
125,unauthorized,mgmt.list_principal_roles,grant_records,"constraint_name, entities_pkey, idx_entities",7,0
126,unauthorized,mgmt.list_principals,grant_records,"constraint_name, entities_pkey, idx_entities",7,0
127,unauthorized,mgmt.list_principals_for_principal_role,grant_records,"constraint_name, entities_pkey, idx_entities",8,0


In [18]:
# --- cell 9: teardown ------------------------------------------------------
#: Notebooks here close what they opened. This one shells out rather than
#: holding a pool, so there is nothing to dispose — but the dataframes are
#: dropped explicitly, per the repo convention.
for _n in ("df", "cdf"):
    if _n in dir():
        del globals()[_n]
gc.collect()
print("done. Index state:", "present" if index_present() else "absent")


done. Index state: absent
